In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from timm.models.swin_transformer import swin_tiny_patch4_window7_224 as SwinTransformer


class MultiScaleAttentionFusion(nn.Module):
    def __init__(self, in_channels):
        super(MultiScaleAttentionFusion, self).__init__()

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, in_channels // 4, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 4, in_channels, 1, bias=False),
            nn.Sigmoid()
        )

        self.spatial_attn = nn.Sequential(
            nn.Conv2d(in_channels, 1, kernel_size=7, padding=3, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        c_attn = self.channel_attn(x)
        x = x * c_attn
        s_attn = self.spatial_attn(x)
        x = x * s_attn
        return x


class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super(UpBlock, self).__init__()
        self.upsample = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2),
        )
        self.conv = nn.Sequential(
            nn.Conv2d(out_channels + skip_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = self.upsample(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class SwinUNet(nn.Module):
    def __init__(self, img_size=224, in_channels=4, out_channels=4):
        super(SwinUNet, self).__init__()

        self.swin = SwinTransformer(pretrained=True)
        self.swin.head = nn.Identity()
        self.swin.patch_embed.proj = nn.Conv2d(in_channels, 96, kernel_size=4, stride=4, padding=0)

        self.attn1 = MultiScaleAttentionFusion(768)
        self.attn2 = MultiScaleAttentionFusion(384)
        self.attn3 = MultiScaleAttentionFusion(192)
        self.attn4 = MultiScaleAttentionFusion(96)

        self.up1 = UpBlock(768, 384, 384)
        self.up2 = UpBlock(384, 192, 192)
        self.up3 = UpBlock(192, 96, 96)
        self.up4 = nn.Sequential(
            nn.ConvTranspose2d(96, out_channels, kernel_size=2, stride=2)
        )

        # Deep supervision outputs
        self.aux_out1 = nn.Conv2d(384, out_channels, kernel_size=1)
        self.aux_out2 = nn.Conv2d(192, out_channels, kernel_size=1)

    def forward(self, x):
        B, C, H, W = x.shape

        x = self.swin.patch_embed(x)
        x = self.swin.patch_embed.norm(x)
        x = self.swin.layers[0](x)
        x1 = x.permute(0, 3, 1, 2)
        x = self.swin.layers[1](x)
        x2 = x.permute(0, 3, 1, 2)
        x = self.swin.layers[2](x)
        x3 = x.permute(0, 3, 1, 2)
        x = self.swin.layers[3](x)
        x4 = x.permute(0, 3, 1, 2)

        # Apply attention to skip features
        x1 = self.attn4(x1)
        x2 = self.attn3(x2)
        x3 = self.attn2(x3)
        x4 = self.attn1(x4)

        # Decoder with skip connections
        x = self.up1(x4, x3)
        aux1 = self.aux_out1(x)
        x = self.up2(x, x2)
        aux2 = self.aux_out2(x)
        x = self.up3(x, x1)
        x = self.up4(x)

        return x, aux1, aux2


# Loss Functions
class SoftDiceLoss(nn.Module):
    def __init__(self, smooth=1.):
        super(SoftDiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        # Resize target if shape mismatch
        if pred.shape[2:] != target.shape[1:]:
            target = F.interpolate(target.unsqueeze(1).float(), size=pred.shape[2:], mode="nearest").squeeze(1).long()

        # Assert class values are within range
        assert target.max() < pred.shape[1], f"Target has class {target.max().item()} >= num_classes {pred.shape[1]}"

        pred = torch.softmax(pred, dim=1)
        target_one_hot = F.one_hot(target, num_classes=pred.shape[1]).permute(0, 3, 1, 2).float()

        dims = (0, 2, 3)
        intersection = (pred * target_one_hot).sum(dim=dims)
        union = pred.sum(dim=dims) + target_one_hot.sum(dim=dims)
        dice = (2. * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()


import torch.nn.functional as F

def combined_loss(preds, target):
    main_pred, aux1, aux2 = preds

    # Resize target to match the main_pred size (112x112)
    target_resized = F.interpolate(target.unsqueeze(1).float(), size=main_pred.shape[2:], mode="nearest").squeeze(1).long()

    # Debugging: Ensure target values are within the expected range
    assert torch.min(target_resized) >= 0 and torch.max(target_resized) < 4, "Target values must be in the range [0, 3]"
    print(f"Resized target min/max: {torch.min(target_resized)} / {torch.max(target_resized)}")
    
    # Dice + CrossEntropy for main prediction
    dice_loss = SoftDiceLoss()(main_pred, target_resized)
    ce_loss = nn.CrossEntropyLoss(
        weight=torch.tensor([0.1, 1.0, 1.0, 3.0]).to(main_pred.device)
    )(main_pred, target_resized)

    # Resize target to match aux1 and aux2 sizes
    target_aux1 = F.interpolate(target.unsqueeze(1).float(), size=aux1.shape[2:], mode="nearest").squeeze(1).long()
    target_aux2 = F.interpolate(target.unsqueeze(1).float(), size=aux2.shape[2:], mode="nearest").squeeze(1).long()

    # Auxiliary losses
    aux1_loss = nn.CrossEntropyLoss()(aux1, target_aux1)
    aux2_loss = nn.CrossEntropyLoss()(aux2, target_aux2)

    # Total loss: Weighted sum
    total_loss = dice_loss + ce_loss + 0.5 * aux1_loss + 0.5 * aux2_loss
    return total_loss



# Test
if __name__ == "__main__":
    device = "cpu"
    model = SwinUNet().to(device)
    dummy_input = torch.randn(2, 4, 224, 224).to(device)
    outputs = model(dummy_input)
    print("✅ Refactored Swin-UNet Forward Pass Successful! Output shape:", outputs[0].shape)

c:\Users\Dell\Desktop\Group_4_sem_6_AIML\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Refactored Swin-UNet Forward Pass Successful! Output shape: torch.Size([2, 4, 112, 112])


In [5]:
def convert_bn_to_gn(model, default_num_groups=32):
    for name, module in model.named_children():
        if isinstance(module, nn.BatchNorm2d):
            num_channels = module.num_features
            # Find the largest divisor of num_channels not exceeding default_num_groups
            num_groups = min(default_num_groups, num_channels)
            while num_channels % num_groups != 0:
                num_groups -= 1
            gn = nn.GroupNorm(num_groups=num_groups, num_channels=num_channels)
            setattr(model, name, gn)
        else:
            convert_bn_to_gn(module, default_num_groups)


In [7]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import numpy as np
from sklearn.preprocessing import label_binarize
import segmentation_models_pytorch as smp

# 1. Load your models (Example: DeepLabV3+ with ResNet50, U-Net with ResNet34, and another model)
device = torch.device("cpu")

# Initialize models
model1 = smp.DeepLabV3Plus(encoder_name="resnet50", encoder_weights="imagenet", in_channels=3, classes=4).to(device)
convert_bn_to_gn(model1)
model1.load_state_dict(torch.load(r"C:\Users\Dell\Desktop\Group_4_sem_6_AIML\comparativeAnalysisModel\DeepLabV3 + ResNet50\saved_models\2019_lgg_deeplabv3_resNet50.pth"))


model2 = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=4).to(device)
model2.load_state_dict(torch.load(r"C:\Users\Dell\Desktop\Group_4_sem_6_AIML\comparativeAnalysisModel\UNet + ResNet34\saved_model\preTrained_unet_resnet34_2019_lgg.pth"))

# For the custom model, you can load it in a similar way (example)
model.load_state_dict(torch.load(r"C:\Users\Dell\Desktop\Group_4_sem_6_AIML\saved model\2019_combined_loss_finetuned_lgg.pth"))


# Set models to evaluation mode
model1.eval()
model2.eval()
model.eval()

# 2. Define the evaluate function to get predicted labels for each model
def get_model_predictions(model, val_loader, device):
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            main_pred = outputs[0]  # Assuming model outputs (logits) or use softmax to get probabilities
            main_pred = F.interpolate(main_pred, size=masks.shape[1:], mode="bilinear", align_corners=False)
            preds = torch.argmax(main_pred, dim=1)  # Get predicted class labels
            all_preds.append(preds.cpu().numpy())
            all_labels.append(masks.cpu().numpy())

    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    return all_preds, all_labels

# Get predictions and true labels from all models
val_loader = ...  # Assuming you have a DataLoader for validation dataset
preds_model1, labels_model1 = get_model_predictions(model1, val_loader, device)
preds_model2, labels_model2 = get_model_predictions(model2, val_loader, device)
preds_model3, labels_model3 = get_model_predictions(model3, val_loader, device)

# 3. Binarize the masks and predictions (for multi-class ROC)
labels_binarized = label_binarize(labels_model1.ravel(), classes=[0, 1, 2, 3])  # Change according to your number of classes
preds_model1_binarized = label_binarize(preds_model1.ravel(), classes=[0, 1, 2, 3])
preds_model2_binarized = label_binarize(preds_model2.ravel(), classes=[0, 1, 2, 3])
preds_model3_binarized = label_binarize(preds_model3.ravel(), classes=[0, 1, 2, 3])

# 4. Compute ROC curve and AUC for each model
fpr_model1, tpr_model1, _ = roc_curve(labels_binarized.ravel(), preds_model1_binarized.ravel())
fpr_model2, tpr_model2, _ = roc_curve(labels_binarized.ravel(), preds_model2_binarized.ravel())
fpr_model3, tpr_model3, _ = roc_curve(labels_binarized.ravel(), preds_model3_binarized.ravel())

roc_auc_model1 = auc(fpr_model1, tpr_model1)
roc_auc_model2 = auc(fpr_model2, tpr_model2)
roc_auc_model3 = auc(fpr_model3, tpr_model3)

# 5. Plot the ROC curve for all models
plt.figure(figsize=(10, 8))
plt.plot(fpr_model1, tpr_model1, color='blue', lw=2, label=f'Model1 (AUC = {roc_auc_model1:.2f})')
plt.plot(fpr_model2, tpr_model2, color='green', lw=2, label=f'Model2 (AUC = {roc_auc_model2:.2f})')
plt.plot(fpr_model3, tpr_model3, color='red', lw=2, label=f'Model3 (AUC = {roc_auc_model3:.2f})')

# 6. Customize the plot
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')  # Diagonal line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


TypeError: 'ellipsis' object is not iterable